# 03b — Interaction-aware personalized valve durability model

This notebook keeps the same pre-operative inputs and grouped train/validation/test split as `03`, but adds two upgrades targeted at **personalized valve comparison**:

```text
longitudinal labs ───────→ Lab GRU ───────┐
                                           │
longitudinal medications → Med GRU ───────┤
                                           ├→ Patient representation z_patient
prior valve history ─────→ History MLP ───┘

candidate valve ─────────→ Candidate MLP ─────→ z_valve

z_patient
z_valve
z_patient ⊙ z_valve
|z_patient - z_valve|
             │
             ↓
      interaction fusion
             ↓
      discrete-time survival
             ↓
        durability years
```

Training uses the joint objective

\[
\mathcal{L}=\mathcal{L}_{survival}+\lambda\mathcal{L}_{ranking}
\]

The ranking term compares **candidate valves within the same patient** and only uses pairs whose observed survival/censoring establishes an ordering. Generator-only counterfactual truth remains hidden until evaluation.

This is a semi-synthetic proof-of-concept, not a clinically validated treatment recommendation system.

In [1]:
from pathlib import Path
import copy
import json
import math
import random

import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 20260917

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

DATA_DIR = Path("synthetic_heterogeneous_preop_v5")

TRAJECTORY_CSV = DATA_DIR / "synthetic_preop_trajectory_v5_MODEL_READY.csv"
SCENARIO_CSV = DATA_DIR / "synthetic_candidate_scenarios_v5_MODEL_READY.csv"
SCENARIO_FULL_CSV = DATA_DIR / "synthetic_candidate_scenarios_v5_FULL.csv"

required = [TRAJECTORY_CSV, SCENARIO_CSV, SCENARIO_FULL_CSV]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required input(s):\n  - "
        + "\n  - ".join(missing)
        + "\nRun 02e first."
    )

HORIZON_YEARS = 12
BIN_WIDTH_YEARS = 1.0
N_BINS = int(HORIZON_YEARS / BIN_WIDTH_YEARS)

BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

# Interaction/ranking upgrade
PATIENTS_PER_BATCH = 16
RANK_LOSS_WEIGHT = 0.30
RANK_TEMPERATURE_YEARS = 0.50

Device: cuda


In [2]:
trajectory_df = pd.read_csv(TRAJECTORY_CSV)
scenario_df = pd.read_csv(SCENARIO_CSV)
scenario_full_df = pd.read_csv(SCENARIO_FULL_CSV)

LAB_COLS = sorted([
    c for c in trajectory_df.columns
    if c.startswith("lab__")
])

MED_COLS = sorted([
    c for c in trajectory_df.columns
    if c.startswith("med__") and c.endswith("__present")
])

HISTORY_COLS = [
    "prior_valve_count",
    "prior_has_SAVR",
    "prior_has_TAVR",
    "prior_redo_count",
    "prior_ViV_count",
    "prior_latest_valve_size_mm",
    "prior_known_model_count",
]

CANDIDATE_CAT_COLS = [
    "candidate_procedure_type",
    "candidate_valve_model",
    "candidate_valve_position",
]

CANDIDATE_NUM_COLS = [
    "candidate_valve_size_mm",
]

assert (trajectory_df["time_from_implant_months"] < 0).all()
assert not any(c.startswith("generator_") for c in trajectory_df.columns)
assert not any(c.startswith("generator_") for c in scenario_df.columns)
assert scenario_df[["Patient", "candidate_id"]].duplicated().sum() == 0

print("Trajectory rows:", len(trajectory_df))
print("Scenario rows:", len(scenario_df))
print("Synthetic patients:", scenario_df["Patient"].nunique())
print("Labs:", len(LAB_COLS))
print("Medications:", len(MED_COLS))
print("Candidate models:", scenario_df["candidate_valve_model"].nunique())

display(scenario_df.head())

Trajectory rows: 4989
Scenario rows: 4000
Synthetic patients: 1000
Labs: 30
Medications: 15
Candidate models: 12


,Patient,counterfactual_group_id,candidate_id,candidate_procedure_type,candidate_valve_model,candidate_valve_size_mm,candidate_valve_position,candidate_observed_frequency,prior_valve_count,prior_has_SAVR,prior_has_TAVR,prior_redo_count,prior_ViV_count,prior_latest_valve_size_mm,prior_known_model_count,duration_months,event,event_type
0,SYN_00001,SYN_00001,OBS_005,SAVR,Carpentier-Edwards bovine pericardial aortic v...,23.0,aortic,1,1,1,0,0,0,23.0,1,83.510717,1,prosthetic_failure
1,SYN_00001,SYN_00001,OBS_013,TAVR,Edwards-Sapien,29.0,aortic,1,1,1,0,0,0,23.0,1,64.510074,1,reintervention
2,SYN_00001,SYN_00001,OBS_003,SAVR,Carpentier-Edwards bovine pericardial,21.0,aortic,1,1,1,0,0,0,23.0,1,86.204403,1,reintervention
3,SYN_00001,SYN_00001,OBS_001,SAVR,23 TRIFECTA,NaN,aortic,1,1,1,0,0,0,23.0,1,83.136488,1,prosthetic_failure
4,SYN_00002,SYN_00002,OBS_006,SAVR,Carpentier-Edwards pericardial,25.0,aortic,1,1,1,0,0,0,23.0,1,89.369762,1,prosthetic_failure


In [3]:
# ============================================================
# GROUPED train / validation / test split
# ============================================================

patient_table = (
    scenario_df[["Patient"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED,
)

trainval_idx, test_idx = next(
    gss_test.split(
        patient_table,
        groups=patient_table["Patient"],
    )
)

trainval_patients_arr = patient_table.iloc[trainval_idx]["Patient"].to_numpy()
test_patients_arr = patient_table.iloc[test_idx]["Patient"].to_numpy()

trainval_table = pd.DataFrame({"Patient": trainval_patients_arr})

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1765,
    random_state=SEED + 1,
)

train_idx, val_idx = next(
    gss_val.split(
        trainval_table,
        groups=trainval_table["Patient"],
    )
)

train_patients = set(trainval_table.iloc[train_idx]["Patient"])
val_patients = set(trainval_table.iloc[val_idx]["Patient"])
test_patients = set(test_patients_arr)

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

def subset_scenarios(patient_set):
    return (
        scenario_df[
            scenario_df["Patient"].isin(patient_set)
        ]
        .copy()
        .reset_index(drop=True)
    )

train_scen = subset_scenarios(train_patients)
val_scen = subset_scenarios(val_patients)
test_scen = subset_scenarios(test_patients)

print("Patients:", len(train_patients), len(val_patients), len(test_patients))
print("Scenarios:", len(train_scen), len(val_scen), len(test_scen))

Patients: 699 151 150
Scenarios: 2796 604 600


In [4]:
# ============================================================
# Fit preprocessing ONLY on training patients
# ============================================================

train_traj = trajectory_df[
    trajectory_df["Patient"].isin(train_patients)
].copy()

lab_mean = (
    train_traj[LAB_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
)

lab_std = (
    train_traj[LAB_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .std()
    .replace(0, 1.0)
    .fillna(1.0)
)

history_mean = (
    train_scen[HISTORY_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
)

history_std = (
    train_scen[HISTORY_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .std()
    .replace(0, 1.0)
    .fillna(1.0)
)

candidate_num_mean = (
    train_scen[CANDIDATE_NUM_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
)

candidate_num_std = (
    train_scen[CANDIDATE_NUM_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .std()
    .replace(0, 1.0)
    .fillna(1.0)
)

def build_vocab(series):
    values = (
        series
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .replace("", "UNKNOWN")
        .unique()
        .tolist()
    )

    values = sorted(set(values) - {"UNKNOWN"})

    vocab = {"UNKNOWN": 0}

    for i, value in enumerate(values, start=1):
        vocab[value] = i

    return vocab

candidate_vocabs = {
    col: build_vocab(train_scen[col])
    for col in CANDIDATE_CAT_COLS
}

for col, vocab in candidate_vocabs.items():
    print(col, "vocab size =", len(vocab))

candidate_procedure_type vocab size = 3
candidate_valve_model vocab size = 13
candidate_valve_position vocab size = 2


In [5]:
# ============================================================
# Transform longitudinal patient trajectories
# ============================================================

trajectory_cache = {}

def transform_patient_trajectory(patient_id):
    if patient_id in trajectory_cache:
        return trajectory_cache[patient_id]

    g = (
        trajectory_df[
            trajectory_df["Patient"].eq(patient_id)
        ]
        .sort_values("time_from_implant_months")
        .copy()
    )

    if len(g) == 0:
        raise KeyError(f"No trajectory for {patient_id}")

    lab_raw = g[LAB_COLS].apply(pd.to_numeric, errors="coerce")
    lab_mask = lab_raw.notna().astype(np.float32)
    lab_z = ((lab_raw - lab_mean) / lab_std).fillna(0.0).astype(np.float32)

    med_raw = g[MED_COLS].apply(pd.to_numeric, errors="coerce")
    med_mask = med_raw.notna().astype(np.float32)
    med_filled = med_raw.fillna(0.0).astype(np.float32)

    time_channel = (
        g["time_from_implant_months"]
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
        / 120.0
    ).astype(np.float32)

    lab_x = np.concatenate(
        [
            lab_z.to_numpy(),
            lab_mask.to_numpy(),
            time_channel,
        ],
        axis=1,
    ).astype(np.float32)

    med_x = np.concatenate(
        [
            med_filled.to_numpy(),
            med_mask.to_numpy(),
            time_channel,
        ],
        axis=1,
    ).astype(np.float32)

    trajectory_cache[patient_id] = {
        "lab": torch.tensor(lab_x, dtype=torch.float32),
        "med": torch.tensor(med_x, dtype=torch.float32),
    }

    return trajectory_cache[patient_id]

LAB_INPUT_DIM = len(LAB_COLS) * 2 + 1
MED_INPUT_DIM = len(MED_COLS) * 2 + 1

print("Lab GRU input dim:", LAB_INPUT_DIM)
print("Medication GRU input dim:", MED_INPUT_DIM)

Lab GRU input dim: 61
Medication GRU input dim: 31


In [6]:
# ============================================================
# Static history + candidate transforms
# ============================================================

def transform_history(row):
    raw = pd.to_numeric(
        row[HISTORY_COLS],
        errors="coerce",
    )

    mask = raw.notna().astype(np.float32)

    z = (
        (raw - history_mean)
        / history_std
    ).fillna(0.0).astype(np.float32)

    x = np.concatenate([
        z.to_numpy(),
        mask.to_numpy(),
    ]).astype(np.float32)

    return torch.tensor(x, dtype=torch.float32)

def encode_category(value, vocab):
    if pd.isna(value):
        return 0

    value = str(value).strip()

    if not value:
        return 0

    return int(vocab.get(value, 0))

def transform_candidate(row):
    cat = {
        col: encode_category(
            row[col],
            candidate_vocabs[col],
        )
        for col in CANDIDATE_CAT_COLS
    }

    raw_num = pd.to_numeric(
        row[CANDIDATE_NUM_COLS],
        errors="coerce",
    )

    num_mask = raw_num.notna().astype(np.float32)

    num_z = (
        (raw_num - candidate_num_mean)
        / candidate_num_std
    ).fillna(0.0).astype(np.float32)

    numeric = np.concatenate([
        num_z.to_numpy(),
        num_mask.to_numpy(),
    ]).astype(np.float32)

    return {
        "procedure": cat["candidate_procedure_type"],
        "model": cat["candidate_valve_model"],
        "position": cat["candidate_valve_position"],
        "numeric": torch.tensor(numeric, dtype=torch.float32),
    }

HISTORY_INPUT_DIM = len(HISTORY_COLS) * 2
CANDIDATE_NUM_INPUT_DIM = len(CANDIDATE_NUM_COLS) * 2

print("History MLP input dim:", HISTORY_INPUT_DIM)
print("Candidate numeric input dim:", CANDIDATE_NUM_INPUT_DIM)

History MLP input dim: 14
Candidate numeric input dim: 2


In [7]:
# ============================================================
# Dataset + PATIENT-GROUPED batch sampler
# ============================================================

class ValveScenarioDataset(Dataset):

    def __init__(self, scenarios):
        self.df = scenarios.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient = row["Patient"]
        traj = transform_patient_trajectory(patient)
        candidate = transform_candidate(row)

        return {
            "Patient": patient,
            "candidate_id": row["candidate_id"],
            "lab": traj["lab"],
            "med": traj["med"],
            "history": transform_history(row),
            "candidate_procedure": candidate["procedure"],
            "candidate_model": candidate["model"],
            "candidate_position": candidate["position"],
            "candidate_numeric": candidate["numeric"],
            "duration_months": float(row["duration_months"]),
            "event": float(row["event"]),
        }


def collate_scenarios(batch):
    lab_lengths = torch.tensor(
        [item["lab"].shape[0] for item in batch],
        dtype=torch.long,
    )
    med_lengths = torch.tensor(
        [item["med"].shape[0] for item in batch],
        dtype=torch.long,
    )

    lab_padded = pad_sequence(
        [item["lab"] for item in batch],
        batch_first=True,
        padding_value=0.0,
    )
    med_padded = pad_sequence(
        [item["med"] for item in batch],
        batch_first=True,
        padding_value=0.0,
    )

    return {
        "Patient": [item["Patient"] for item in batch],
        "candidate_id": [item["candidate_id"] for item in batch],
        "lab": lab_padded,
        "lab_lengths": lab_lengths,
        "med": med_padded,
        "med_lengths": med_lengths,
        "history": torch.stack([item["history"] for item in batch]),
        "candidate_procedure": torch.tensor(
            [item["candidate_procedure"] for item in batch],
            dtype=torch.long,
        ),
        "candidate_model": torch.tensor(
            [item["candidate_model"] for item in batch],
            dtype=torch.long,
        ),
        "candidate_position": torch.tensor(
            [item["candidate_position"] for item in batch],
            dtype=torch.long,
        ),
        "candidate_numeric": torch.stack(
            [item["candidate_numeric"] for item in batch]
        ),
        "duration_months": torch.tensor(
            [item["duration_months"] for item in batch],
            dtype=torch.float32,
        ),
        "event": torch.tensor(
            [item["event"] for item in batch],
            dtype=torch.float32,
        ),
    }


class PatientGroupedBatchSampler:
    # Keeps every candidate scenario for a patient in the same minibatch.

    def __init__(
        self,
        dataset,
        patients_per_batch=16,
        shuffle=True,
        seed=SEED,
    ):
        self.dataset = dataset
        self.patients_per_batch = int(patients_per_batch)
        self.shuffle = bool(shuffle)
        self.seed = int(seed)
        self.epoch = 0

        patient_to_indices = {}
        for idx, patient in enumerate(dataset.df["Patient"].tolist()):
            patient_to_indices.setdefault(patient, []).append(idx)

        self.patient_to_indices = patient_to_indices
        self.patients = list(patient_to_indices.keys())

    def __iter__(self):
        patients = list(self.patients)

        if self.shuffle:
            local_rng = np.random.default_rng(self.seed + self.epoch)
            local_rng.shuffle(patients)

        self.epoch += 1

        for start in range(0, len(patients), self.patients_per_batch):
            selected = patients[start:start + self.patients_per_batch]
            batch_indices = []
            for patient in selected:
                batch_indices.extend(self.patient_to_indices[patient])
            yield batch_indices

    def __len__(self):
        return int(math.ceil(len(self.patients) / self.patients_per_batch))


train_ds = ValveScenarioDataset(train_scen)
val_ds = ValveScenarioDataset(val_scen)
test_ds = ValveScenarioDataset(test_scen)

train_loader = DataLoader(
    train_ds,
    batch_sampler=PatientGroupedBatchSampler(
        train_ds,
        patients_per_batch=PATIENTS_PER_BATCH,
        shuffle=True,
        seed=SEED,
    ),
    collate_fn=collate_scenarios,
)

val_loader = DataLoader(
    val_ds,
    batch_sampler=PatientGroupedBatchSampler(
        val_ds,
        patients_per_batch=PATIENTS_PER_BATCH,
        shuffle=False,
        seed=SEED + 1,
    ),
    collate_fn=collate_scenarios,
)

test_loader = DataLoader(
    test_ds,
    batch_sampler=PatientGroupedBatchSampler(
        test_ds,
        patients_per_batch=PATIENTS_PER_BATCH,
        shuffle=False,
        seed=SEED + 2,
    ),
    collate_fn=collate_scenarios,
)

batch = next(iter(train_loader))

print("Example grouped batch scenarios:", len(batch["Patient"]))
print("Patients in grouped batch:", len(set(batch["Patient"])))
print("Lab batch:", batch["lab"].shape)
print("Medication batch:", batch["med"].shape)
print("History batch:", batch["history"].shape)

Example grouped batch scenarios: 64
Patients in grouped batch: 16
Lab batch: torch.Size([64, 21, 61])
Medication batch: torch.Size([64, 21, 31])
History batch: torch.Size([64, 14])


In [8]:
# ============================================================
# Interaction-aware model
# ============================================================

class SequenceGRUEncoder(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(),
            nn.LayerNorm(output_dim),
        )

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, h = self.gru(packed)
        return self.proj(h[-1])


class CandidateValveEncoder(nn.Module):

    def __init__(
        self,
        n_procedure,
        n_model,
        n_position,
        numeric_dim,
        output_dim=48,
    ):
        super().__init__()

        self.procedure_emb = nn.Embedding(n_procedure, 4)
        self.model_emb = nn.Embedding(n_model, 12)
        self.position_emb = nn.Embedding(n_position, 3)

        in_dim = 4 + 12 + 3 + numeric_dim

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, output_dim),
            nn.ReLU(),
            nn.LayerNorm(output_dim),
        )

    def forward(self, procedure, model, position, numeric):
        x = torch.cat(
            [
                self.procedure_emb(procedure),
                self.model_emb(model),
                self.position_emb(position),
                numeric,
            ],
            dim=1,
        )
        return self.mlp(x)


class InteractionAwareValveSurvivalModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.lab_encoder = SequenceGRUEncoder(
            input_dim=LAB_INPUT_DIM,
            hidden_dim=64,
            output_dim=32,
        )

        self.med_encoder = SequenceGRUEncoder(
            input_dim=MED_INPUT_DIM,
            hidden_dim=48,
            output_dim=24,
        )

        self.history_encoder = nn.Sequential(
            nn.Linear(HISTORY_INPUT_DIM, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 20),
            nn.ReLU(),
            nn.LayerNorm(20),
        )

        patient_raw_dim = 32 + 24 + 20

        self.patient_projection = nn.Sequential(
            nn.Linear(patient_raw_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 48),
            nn.ReLU(),
            nn.LayerNorm(48),
        )

        self.candidate_encoder = CandidateValveEncoder(
            n_procedure=len(candidate_vocabs["candidate_procedure_type"]),
            n_model=len(candidate_vocabs["candidate_valve_model"]),
            n_position=len(candidate_vocabs["candidate_valve_position"]),
            numeric_dim=CANDIDATE_NUM_INPUT_DIM,
            output_dim=48,
        )

        # patient + valve + multiplicative interaction + distance
        self.interaction_fusion = nn.Sequential(
            nn.Linear(48 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.LayerNorm(64),
        )

        self.survival_head = nn.Linear(64, N_BINS)

    def encode_patient(self, batch):
        z_lab = self.lab_encoder(batch["lab"], batch["lab_lengths"])
        z_med = self.med_encoder(batch["med"], batch["med_lengths"])
        z_history = self.history_encoder(batch["history"])

        z_raw = torch.cat([z_lab, z_med, z_history], dim=1)
        return self.patient_projection(z_raw)

    def encode_candidate(self, batch):
        return self.candidate_encoder(
            batch["candidate_procedure"],
            batch["candidate_model"],
            batch["candidate_position"],
            batch["candidate_numeric"],
        )

    def forward(self, batch, return_embeddings=False):
        z_patient = self.encode_patient(batch)
        z_valve = self.encode_candidate(batch)

        z_product = z_patient * z_valve
        z_absdiff = torch.abs(z_patient - z_valve)

        interaction = torch.cat(
            [z_patient, z_valve, z_product, z_absdiff],
            dim=1,
        )

        fused = self.interaction_fusion(interaction)
        hazard_logits = self.survival_head(fused)

        if return_embeddings:
            return {
                "hazard_logits": hazard_logits,
                "z_patient": z_patient,
                "z_valve": z_valve,
                "z_product": z_product,
            }

        return hazard_logits


model = InteractionAwareValveSurvivalModel().to(DEVICE)
print(model)


def move_batch_to_device(batch):
    moved = dict(batch)

    tensor_keys = [
        "lab",
        "lab_lengths",
        "med",
        "med_lengths",
        "history",
        "candidate_procedure",
        "candidate_model",
        "candidate_position",
        "candidate_numeric",
        "duration_months",
        "event",
    ]

    for key in tensor_keys:
        moved[key] = moved[key].to(DEVICE)

    return moved


with torch.no_grad():
    smoke_batch = move_batch_to_device(batch)
    smoke_outputs = model(smoke_batch, return_embeddings=True)

print("Forward-pass hazards:", smoke_outputs["hazard_logits"].shape)
print("Patient embedding:", smoke_outputs["z_patient"].shape)
print("Valve embedding:", smoke_outputs["z_valve"].shape)

assert smoke_outputs["hazard_logits"].shape[1] == N_BINS

InteractionAwareValveSurvivalModel(
  (lab_encoder): SequenceGRUEncoder(
    (gru): GRU(61, 64, batch_first=True)
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=32, bias=True)
      (1): ReLU()
      (2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    )
  )
  (med_encoder): SequenceGRUEncoder(
    (gru): GRU(31, 48, batch_first=True)
    (proj): Sequential(
      (0): Linear(in_features=48, out_features=24, bias=True)
      (1): ReLU()
      (2): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
    )
  )
  (history_encoder): Sequential(
    (0): Linear(in_features=14, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=20, bias=True)
    (4): ReLU()
    (5): LayerNorm((20,), eps=1e-05, elementwise_affine=True)
  )
  (patient_projection): Sequential(
    (0): Linear(in_features=76, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Li

In [9]:
# ============================================================
# Joint survival + censor-aware within-patient ranking loss
# ============================================================

def discrete_time_survival_nll(
    hazard_logits,
    duration_months,
    event,
):
    duration_years = duration_months / 12.0
    batch_size = hazard_logits.shape[0]

    targets = torch.zeros_like(hazard_logits)
    mask = torch.zeros_like(hazard_logits)

    for i in range(batch_size):
        t = float(duration_years[i].item())
        e = int(event[i].item())

        if e == 1:
            event_bin = int(math.floor(t / BIN_WIDTH_YEARS))
            event_bin = min(max(event_bin, 0), N_BINS - 1)
            mask[i, :event_bin + 1] = 1.0
            targets[i, event_bin] = 1.0
        else:
            completed_bins = int(math.floor(t / BIN_WIDTH_YEARS))
            completed_bins = min(max(completed_bins, 0), N_BINS)
            if completed_bins > 0:
                mask[i, :completed_bins] = 1.0

    bce = nn.functional.binary_cross_entropy_with_logits(
        hazard_logits,
        targets,
        reduction="none",
    )

    denom = mask.sum().clamp_min(1.0)
    return (bce * mask).sum() / denom


def differentiable_rmst_years(hazard_logits):
    hazards = torch.sigmoid(hazard_logits)
    survival = torch.cumprod(1.0 - hazards, dim=1)

    survival_start = torch.cat(
        [
            torch.ones(
                (survival.shape[0], 1),
                device=survival.device,
                dtype=survival.dtype,
            ),
            survival[:, :-1],
        ],
        dim=1,
    )

    return survival_start.sum(dim=1) * BIN_WIDTH_YEARS


def censor_aware_pairwise_ranking_loss(
    predicted_rmst,
    duration_months,
    event,
    patient_ids,
    temperature_years=0.50,
):
    # Uses only observable survival ordering; never hidden generator truth.

    groups = {}
    for idx, patient in enumerate(patient_ids):
        groups.setdefault(patient, []).append(idx)

    losses = []
    comparable_pairs = 0
    duration_years = duration_months / 12.0

    for indices in groups.values():
        if len(indices) < 2:
            continue

        for a in range(len(indices)):
            for b in range(a + 1, len(indices)):
                i = indices[a]
                j = indices[b]

                ti = float(duration_years[i].detach().cpu())
                tj = float(duration_years[j].detach().cpu())
                ei = int(event[i].detach().cpu())
                ej = int(event[j].detach().cpu())

                if ej == 1 and tj < ti:
                    # j failed while i remained event-free -> predict i longer.
                    score_diff = predicted_rmst[i] - predicted_rmst[j]
                    observed_gap = ti - tj

                elif ei == 1 and ti < tj:
                    # i failed while j remained event-free -> predict j longer.
                    score_diff = predicted_rmst[j] - predicted_rmst[i]
                    observed_gap = tj - ti

                else:
                    continue

                comparable_pairs += 1

                pair_weight = 0.5 + min(observed_gap, 3.0) / 3.0

                losses.append(
                    nn.functional.softplus(
                        -score_diff / temperature_years
                    ) * pair_weight
                )

    if not losses:
        return predicted_rmst.sum() * 0.0, 0

    return torch.stack(losses).mean(), comparable_pairs


survival_loss_smoke = discrete_time_survival_nll(
    smoke_outputs["hazard_logits"],
    smoke_batch["duration_months"],
    smoke_batch["event"],
)

smoke_rmst = differentiable_rmst_years(
    smoke_outputs["hazard_logits"]
)

rank_loss_smoke, n_pairs_smoke = censor_aware_pairwise_ranking_loss(
    smoke_rmst,
    smoke_batch["duration_months"],
    smoke_batch["event"],
    batch["Patient"],
    temperature_years=RANK_TEMPERATURE_YEARS,
)

print("Survival loss smoke test:", float(survival_loss_smoke))
print("Ranking loss smoke test:", float(rank_loss_smoke))
print("Comparable within-patient pairs:", n_pairs_smoke)

Survival loss smoke test: 0.7492442727088928
Ranking loss smoke test: 0.47852063179016113
Comparable within-patient pairs: 78


In [10]:
# ============================================================
# Joint-objective training
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


def run_epoch(model, loader, train=False):

    if train:
        model.train()
    else:
        model.eval()

    total_losses = []
    survival_losses = []
    ranking_losses = []
    pair_counts = []

    for batch in loader:
        patient_ids = list(batch["Patient"])
        batch_dev = move_batch_to_device(batch)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(batch_dev)

            survival_loss = discrete_time_survival_nll(
                logits,
                batch_dev["duration_months"],
                batch_dev["event"],
            )

            predicted_rmst = differentiable_rmst_years(logits)

            ranking_loss, n_pairs = censor_aware_pairwise_ranking_loss(
                predicted_rmst,
                batch_dev["duration_months"],
                batch_dev["event"],
                patient_ids,
                temperature_years=RANK_TEMPERATURE_YEARS,
            )

            total_loss = (
                survival_loss
                + RANK_LOSS_WEIGHT * ranking_loss
            )

        if train:
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        total_losses.append(float(total_loss.detach().cpu()))
        survival_losses.append(float(survival_loss.detach().cpu()))
        ranking_losses.append(float(ranking_loss.detach().cpu()))
        pair_counts.append(int(n_pairs))

    return {
        "total_loss": float(np.mean(total_losses)),
        "survival_nll": float(np.mean(survival_losses)),
        "ranking_loss": float(np.mean(ranking_losses)),
        "comparable_pairs": int(np.sum(pair_counts)),
    }


best_val_loss = np.inf
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):

    train_metrics = run_epoch(model, train_loader, train=True)
    val_metrics = run_epoch(model, val_loader, train=False)

    history.append({
        "epoch": epoch,
        "train_total_loss": train_metrics["total_loss"],
        "train_survival_nll": train_metrics["survival_nll"],
        "train_ranking_loss": train_metrics["ranking_loss"],
        "train_comparable_pairs": train_metrics["comparable_pairs"],
        "val_total_loss": val_metrics["total_loss"],
        "val_survival_nll": val_metrics["survival_nll"],
        "val_ranking_loss": val_metrics["ranking_loss"],
        "val_comparable_pairs": val_metrics["comparable_pairs"],
    })

    val_loss = val_metrics["total_loss"]
    improved = val_loss < best_val_loss - 1e-4

    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0 or improved:
        print(
            f"Epoch {epoch:03d} | "
            f"train total={train_metrics['total_loss']:.4f} "
            f"(surv={train_metrics['survival_nll']:.4f}, "
            f"rank={train_metrics['ranking_loss']:.4f}) | "
            f"val total={val_metrics['total_loss']:.4f} "
            f"(surv={val_metrics['survival_nll']:.4f}, "
            f"rank={val_metrics['ranking_loss']:.4f}) | "
            f"pairs train/val={train_metrics['comparable_pairs']}/"
            f"{val_metrics['comparable_pairs']}"
        )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid model state.")

model.load_state_dict(best_state)

print("Best validation joint loss:", round(best_val_loss, 5))

training_history = pd.DataFrame(history)
display(training_history.tail())

Epoch 001 | train total=0.4703 (surv=0.3305, rank=0.4660) | val total=0.3843 (surv=0.2732, rank=0.3704) | pairs train/val=2966/702
Epoch 002 | train total=0.3441 (surv=0.2464, rank=0.3257) | val total=0.2901 (surv=0.2163, rank=0.2460) | pairs train/val=2966/702
Epoch 003 | train total=0.2821 (surv=0.2056, rank=0.2548) | val total=0.2611 (surv=0.2003, rank=0.2026) | pairs train/val=2966/702
Epoch 004 | train total=0.2513 (surv=0.1905, rank=0.2026) | val total=0.2455 (surv=0.1918, rank=0.1789) | pairs train/val=2966/702
Epoch 005 | train total=0.2389 (surv=0.1800, rank=0.1962) | val total=0.2376 (surv=0.1893, rank=0.1610) | pairs train/val=2966/702
Epoch 006 | train total=0.2273 (surv=0.1754, rank=0.1727) | val total=0.2336 (surv=0.1888, rank=0.1493) | pairs train/val=2966/702
Epoch 007 | train total=0.2253 (surv=0.1735, rank=0.1725) | val total=0.2270 (surv=0.1791, rank=0.1594) | pairs train/val=2966/702
Epoch 008 | train total=0.2202 (surv=0.1718, rank=0.1615) | val total=0.2229 (surv=

,epoch,train_total_loss,train_survival_nll,train_ranking_loss,train_comparable_pairs,val_total_loss,val_survival_nll,val_ranking_loss,val_comparable_pairs
19,20,0.198038,0.162227,0.119369,2966,0.221325,0.185662,0.118880,702
20,21,0.197987,0.162987,0.116666,2966,0.223702,0.186844,0.122861,702
21,22,0.194786,0.159505,0.117601,2966,0.222820,0.187169,0.118837,702
22,23,0.194302,0.160608,0.112311,2966,0.220465,0.182350,0.127050,702
23,24,0.193943,0.160461,0.111605,2966,0.224406,0.188782,0.118748,702


In [11]:
# ============================================================
# Hazards -> survival + durability in years
# ============================================================

def hazards_to_outputs(hazard_logits):

    hazards = torch.sigmoid(hazard_logits)

    survival = torch.cumprod(
        1.0 - hazards,
        dim=1,
    )

    survival_start = torch.cat(
        [
            torch.ones(
                (survival.shape[0], 1),
                device=survival.device,
            ),
            survival[:, :-1],
        ],
        dim=1,
    )

    rmst_years = (
        survival_start.sum(dim=1)
        * BIN_WIDTH_YEARS
    )

    return hazards, survival, rmst_years

def survival_probability_at_year(
    survival,
    year,
):
    idx = int(
        math.ceil(
            year / BIN_WIDTH_YEARS
        ) - 1
    )

    idx = max(
        0,
        min(
            idx,
            survival.shape[1] - 1,
        ),
    )

    return survival[:, idx]

def predict_loader(model, loader):

    model.eval()
    rows = []

    with torch.no_grad():

        for batch in loader:

            metadata_patients = list(
                batch["Patient"]
            )

            metadata_candidates = list(
                batch["candidate_id"]
            )

            batch_dev = move_batch_to_device(batch)

            logits = model(batch_dev)

            hazards, survival, rmst = hazards_to_outputs(
                logits
            )

            surv_5 = survival_probability_at_year(
                survival,
                5,
            )

            surv_8 = survival_probability_at_year(
                survival,
                8,
            )

            surv_10 = survival_probability_at_year(
                survival,
                10,
            )

            for i in range(len(metadata_patients)):
                rows.append({
                    "Patient": metadata_patients[i],
                    "candidate_id": metadata_candidates[i],

                    "duration_months": float(
                        batch["duration_months"][i]
                    ),

                    "event": int(
                        batch["event"][i]
                    ),

                    "predicted_rmst_years": float(
                        rmst[i].cpu()
                    ),

                    "pred_survival_5y": float(
                        surv_5[i].cpu()
                    ),

                    "pred_survival_8y": float(
                        surv_8[i].cpu()
                    ),

                    "pred_survival_10y": float(
                        surv_10[i].cpu()
                    ),
                })

    return pd.DataFrame(rows)

test_predictions = predict_loader(
    model,
    test_loader,
)

display(test_predictions.head())

,Patient,candidate_id,duration_months,event,predicted_rmst_years,pred_survival_5y,pred_survival_8y,pred_survival_10y
0,SYN_00028,OBS_002,103.888885,0,8.985411,0.986695,0.578811,0.128671
1,SYN_00028,OBS_007,103.888885,0,8.594035,0.975849,0.489897,0.074720
2,SYN_00028,OBS_005,103.888885,0,8.831663,0.983701,0.542786,0.102048
3,SYN_00028,OBS_008,103.888885,0,9.733982,0.991515,0.756723,0.315300
4,SYN_00030,OBS_006,95.914642,1,8.765347,0.981382,0.553252,0.105159


In [12]:
# ============================================================
# Ordinary survival discrimination
# ============================================================

def harrell_c_index(
    duration_months,
    event,
    predicted_durability,
):
    t = np.asarray(
        duration_months,
        dtype=float,
    )

    e = np.asarray(
        event,
        dtype=int,
    )

    pred = np.asarray(
        predicted_durability,
        dtype=float,
    )

    concordant = 0.0
    comparable = 0.0

    n = len(t)

    for i in range(n):

        if e[i] != 1:
            continue

        for j in range(n):

            if i == j:
                continue

            if t[i] < t[j]:

                comparable += 1.0

                if pred[i] < pred[j]:
                    concordant += 1.0
                elif pred[i] == pred[j]:
                    concordant += 0.5

    if comparable == 0:
        return np.nan

    return concordant / comparable

c_index = harrell_c_index(
    test_predictions["duration_months"],
    test_predictions["event"],
    test_predictions["predicted_rmst_years"],
)

print(
    "Test Harrell C-index:",
    round(float(c_index), 4),
)

Test Harrell C-index: 0.8628


In [13]:
# ============================================================
# Counterfactual ranking recovery
#
# Hidden generator truth is used ONLY for evaluation.
# ============================================================

truth_cols = [
    "Patient",
    "candidate_id",
    "candidate_procedure_type",
    "candidate_valve_model",
    "candidate_valve_size_mm",
    "generator_expected_durability_years",
]

hidden_truth = scenario_full_df[
    truth_cols
].copy()

cf_eval = (
    test_predictions
    .merge(
        hidden_truth,
        on=[
            "Patient",
            "candidate_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert (
    cf_eval[
        "generator_expected_durability_years"
    ]
    .notna()
    .all()
)

def pairwise_ranking_accuracy(
    true_values,
    pred_values,
):
    true_values = np.asarray(
        true_values,
        dtype=float,
    )

    pred_values = np.asarray(
        pred_values,
        dtype=float,
    )

    correct = 0.0
    total = 0.0

    n = len(true_values)

    for i in range(n):
        for j in range(i + 1, n):

            true_diff = (
                true_values[i]
                - true_values[j]
            )

            pred_diff = (
                pred_values[i]
                - pred_values[j]
            )

            if true_diff == 0:
                continue

            total += 1.0

            if np.sign(true_diff) == np.sign(pred_diff):
                correct += 1.0
            elif pred_diff == 0:
                correct += 0.5

    return (
        correct / total
        if total > 0
        else np.nan
    )

patient_metrics = []

for patient, g in cf_eval.groupby("Patient"):

    if len(g) < 2:
        continue

    truth = (
        g["generator_expected_durability_years"]
        .to_numpy()
    )

    pred = (
        g["predicted_rmst_years"]
        .to_numpy()
    )

    rho = spearmanr(
        truth,
        pred,
    ).statistic

    true_best = g.loc[
        g[
            "generator_expected_durability_years"
        ].idxmax(),
        "candidate_id",
    ]

    pred_best = g.loc[
        g[
            "predicted_rmst_years"
        ].idxmax(),
        "candidate_id",
    ]

    patient_metrics.append({
        "Patient": patient,
        "n_candidates": len(g),
        "spearman": rho,
        "top1_correct": int(
            true_best == pred_best
        ),
        "pairwise_accuracy":
            pairwise_ranking_accuracy(
                truth,
                pred,
            ),
    })

patient_cf_metrics = pd.DataFrame(
    patient_metrics
)

print("Counterfactual ranking metrics")

print(
    "Mean within-patient Spearman:",
    round(
        float(
            patient_cf_metrics[
                "spearman"
            ]
            .dropna()
            .mean()
        ),
        4,
    ),
)

print(
    "Top-1 candidate accuracy:",
    round(
        float(
            patient_cf_metrics[
                "top1_correct"
            ].mean()
        ),
        4,
    ),
)

print(
    "Pairwise candidate ranking accuracy:",
    round(
        float(
            patient_cf_metrics[
                "pairwise_accuracy"
            ]
            .dropna()
            .mean()
        ),
        4,
    ),
)

global_spearman = spearmanr(
    cf_eval[
        "generator_expected_durability_years"
    ],
    cf_eval[
        "predicted_rmst_years"
    ],
).statistic

global_pearson = pearsonr(
    cf_eval[
        "generator_expected_durability_years"
    ],
    cf_eval[
        "predicted_rmst_years"
    ],
).statistic

print(
    "Global hidden-truth Spearman:",
    round(float(global_spearman), 4),
)

print(
    "Global hidden-truth Pearson:",
    round(float(global_pearson), 4),
)

display(patient_cf_metrics.head())

Counterfactual ranking metrics
Mean within-patient Spearman: 0.8227
Top-1 candidate accuracy: 0.76
Pairwise candidate ranking accuracy: 0.8833
Global hidden-truth Spearman: 0.9083
Global hidden-truth Pearson: 0.9417


,Patient,n_candidates,spearman,top1_correct,pairwise_accuracy
0,SYN_00028,4,0.8,1,0.833333
1,SYN_00030,4,1.0,1,1.000000
2,SYN_00043,4,0.4,0,0.666667
3,SYN_00050,4,1.0,1,1.000000
4,SYN_00058,4,1.0,1,1.000000


In [14]:
# ============================================================
# Held-out patient candidate comparison
# ============================================================

audit_patient = sorted(test_patients)[0]

audit = (
    cf_eval[
        cf_eval["Patient"].eq(
            audit_patient
        )
    ]
    .copy()
)

audit["prediction_error_years"] = (
    audit["predicted_rmst_years"]
    - audit[
        "generator_expected_durability_years"
    ]
)

audit = audit.sort_values(
    "predicted_rmst_years",
    ascending=False,
)

print("Held-out patient:", audit_patient)

display(
    audit[
        [
            "candidate_id",
            "candidate_procedure_type",
            "candidate_valve_model",
            "candidate_valve_size_mm",
            "generator_expected_durability_years",
            "predicted_rmst_years",
            "pred_survival_5y",
            "pred_survival_8y",
            "pred_survival_10y",
            "prediction_error_years",
        ]
    ]
)

Held-out patient: SYN_00028


,candidate_id,candidate_procedure_type,candidate_valve_model,candidate_valve_size_mm,generator_expected_durability_years,predicted_rmst_years,pred_survival_5y,pred_survival_8y,pred_survival_10y,prediction_error_years
3,OBS_008,SAVR,Perimount,23.0,8.917237,9.733982,0.991515,0.756723,0.315300,0.816745
0,OBS_002,SAVR,23-mm pericardial prosthesis,23.0,8.627011,8.985411,0.986695,0.578811,0.128671,0.358399
2,OBS_005,SAVR,Carpentier-Edwards bovine pericardial aortic v...,23.0,8.507841,8.831663,0.983701,0.542786,0.102048,0.323822
1,OBS_007,SAVR,Carpentier-Edwards prosthetic aortic valve (si...,25.0,8.515431,8.594035,0.975849,0.489897,0.074720,0.078604


In [15]:
# ============================================================
# Save model + preprocessing + predictions
# ============================================================

MODEL_DIR = Path(
    "preop_valve_survival_model_03b"
)

MODEL_DIR.mkdir(
    exist_ok=True
)

model_path = (
    MODEL_DIR
    / "personalized_valve_survival_model.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "n_bins": N_BINS,
        "bin_width_years": BIN_WIDTH_YEARS,
        "rank_loss_weight": RANK_LOSS_WEIGHT,
        "rank_temperature_years": RANK_TEMPERATURE_YEARS,
        "architecture": "interaction_aware_patient_valve",
        "lab_features": LAB_COLS,
        "medication_features": MED_COLS,
        "history_features": HISTORY_COLS,
        "candidate_cat_features": CANDIDATE_CAT_COLS,
        "candidate_num_features": CANDIDATE_NUM_COLS,
        "candidate_vocabs": candidate_vocabs,
    },
    model_path,
)

preprocessing = {
    "lab_mean": lab_mean.to_dict(),
    "lab_std": lab_std.to_dict(),
    "history_mean": history_mean.to_dict(),
    "history_std": history_std.to_dict(),
    "candidate_num_mean": candidate_num_mean.to_dict(),
    "candidate_num_std": candidate_num_std.to_dict(),
    "candidate_vocabs": candidate_vocabs,
    "horizon_years": HORIZON_YEARS,
    "bin_width_years": BIN_WIDTH_YEARS,
    "rank_loss_weight": RANK_LOSS_WEIGHT,
    "rank_temperature_years": RANK_TEMPERATURE_YEARS,
    "architecture": "interaction_aware_patient_valve",
}

preproc_path = (
    MODEL_DIR
    / "preprocessing.json"
)

with open(preproc_path, "w") as f:
    json.dump(
        preprocessing,
        f,
        indent=2,
    )

pred_path = (
    MODEL_DIR
    / "test_candidate_predictions.csv"
)

cf_eval.to_csv(
    pred_path,
    index=False,
)

metric_summary = {
    "test_harrell_c_index":
        float(c_index),

    "mean_within_patient_spearman":
        float(
            patient_cf_metrics[
                "spearman"
            ]
            .dropna()
            .mean()
        ),

    "top1_candidate_accuracy":
        float(
            patient_cf_metrics[
                "top1_correct"
            ]
            .mean()
        ),

    "pairwise_candidate_ranking_accuracy":
        float(
            patient_cf_metrics[
                "pairwise_accuracy"
            ]
            .dropna()
            .mean()
        ),

    "global_hidden_truth_spearman":
        float(global_spearman),

    "global_hidden_truth_pearson":
        float(global_pearson),
}

metrics_path = (
    MODEL_DIR
    / "metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(
        metric_summary,
        f,
        indent=2,
    )

print("✓", model_path)
print("✓", preproc_path)
print("✓", pred_path)
print("✓", metrics_path)

print("\nFinal metrics:")
print(
    json.dumps(
        metric_summary,
        indent=2,
    )
)

# ------------------------------------------------------------
# Optional direct comparison against baseline 03
# ------------------------------------------------------------

baseline_metrics_path = Path("preop_valve_survival_model") / "metrics.json"

if baseline_metrics_path.exists():
    with open(baseline_metrics_path) as f:
        baseline_metrics = json.load(f)

    comparison_rows = []

    for metric in [
        "test_harrell_c_index",
        "mean_within_patient_spearman",
        "top1_candidate_accuracy",
        "pairwise_candidate_ranking_accuracy",
        "global_hidden_truth_spearman",
        "global_hidden_truth_pearson",
    ]:
        old = baseline_metrics.get(metric, np.nan)
        new = metric_summary.get(metric, np.nan)

        comparison_rows.append({
            "metric": metric,
            "baseline_03": old,
            "interaction_03b": new,
            "delta": (
                new - old
                if np.isfinite(old) and np.isfinite(new)
                else np.nan
            ),
        })

    comparison_df = pd.DataFrame(comparison_rows)

    print("\n03b vs baseline 03:")
    display(comparison_df)
else:
    print("\nBaseline metrics.json not found; 03b metrics saved normally.")


✓ preop_valve_survival_model_03b/personalized_valve_survival_model.pt
✓ preop_valve_survival_model_03b/preprocessing.json
✓ preop_valve_survival_model_03b/test_candidate_predictions.csv
✓ preop_valve_survival_model_03b/metrics.json

Final metrics:
{
  "test_harrell_c_index": 0.862787207973678,
  "mean_within_patient_spearman": 0.8226666666666667,
  "top1_candidate_accuracy": 0.76,
  "pairwise_candidate_ranking_accuracy": 0.8833333333333333,
  "global_hidden_truth_spearman": 0.9083178564384901,
  "global_hidden_truth_pearson": 0.9416687753922589
}

03b vs baseline 03:


,metric,baseline_03,interaction_03b,delta
0,test_harrell_c_index,0.878481,0.862787,-0.015694
1,mean_within_patient_spearman,0.438667,0.822667,0.384000
2,top1_candidate_accuracy,0.513333,0.760000,0.246667
3,pairwise_candidate_ranking_accuracy,0.687778,0.883333,0.195556
4,global_hidden_truth_spearman,0.935195,0.908318,-0.026877
5,global_hidden_truth_pearson,0.963669,0.941669,-0.022000


## How to judge whether `03b` helped

Baseline `03` achieved approximately:

- Harrell C-index: **0.878**
- within-patient Spearman: **0.439**
- top-1 candidate accuracy: **0.513**
- pairwise candidate ranking accuracy: **0.688**
- global hidden-truth Spearman: **0.935**

The main goal of `03b` is not necessarily to increase the already-strong global correlation. The desired result is to preserve strong survival discrimination while improving **within-patient candidate ranking**.

That is the part most aligned with the doctor-facing question:

> **For this same patient, how does predicted durability change across candidate valves?**